# Usecase Reprojection

This Jupyter notebook is a usecase created to compare performance and memory usage when doing a Reprojection of a raster.

Here we will compare the Reprojection feature with the following set-ups :
- rasterio
- rioxarray
- rioxarray + dask

In [ ]:
import numpy as np

import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

import rioxarray as rxr

from dask.distributed import Client

from memory_profiler import memory_usage #to compare memory usage

In [2]:
#raster path
raster_path = "../data/rasters/T32UPV_20251201T102411_TCI_10m.jp2"

## Reprojection with rasterio

In [3]:
with rasterio.open(raster_path) as src:
    print(src.crs)

EPSG:32632


The current CRS of our raster is EPSG:32632, and we want to reproject it to EPSG:4326.

We will combine the reproject process and saving the new raster into a single function called *reproj_rasterio* :

In [66]:
dst_crs = "EPSG:4326" #targeted CRS

dst_path = "../outputs/usecase_rasterio_reproj.tif"

def reproj_rasterio():
    with rasterio.open(raster_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy() #copy metadatas
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        with rasterio.open(dst_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1): #band-by-band reprojection
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )

Finally, we can call our new function with memory_usage to track RAM usage, and %%time to see processing time :

In [75]:
%%time
mem_rasterio = memory_usage((reproj_rasterio,))
print(f"RAM max : {max(mem_rasterio):.1f} MB")

RAM max : 1907.6 MB
CPU times: total: 53.2 s
Wall time: 6.89 s


In [76]:
%%timeit
reproj_rasterio()

6.49 s ± 43.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Reprojection with rioxarray

This time we will do the same but using rioxarray. We will create a function combining the reprojection to the new CRS and saving the file, into a single function called *reproj_rioxarray* :

In [70]:
dst_crs = "EPSG:4326"

dst_path = "../outputs/usecase_rxr_reproj.tif"

def reproj_rioxarray():
    ds_rxr = rxr.open_rasterio(raster_path)
    rxr_reproj = ds_rxr.rio.reproject(dst_crs)
    rxr_reproj.rio.to_raster(dst_path)

We call once again our new function with memory_usage to track RAM usage, and %%time to see processing time :

In [71]:
%%time
mem_rioxarray = memory_usage((reproj_rioxarray,))
print(f"RAM max : {max(mem_rioxarray):.1f} MB")

RAM max : 3487.7 MB
CPU times: total: 26.1 s
Wall time: 4.3 s


In [77]:
%%timeit
reproj_rioxarray()

3.82 s ± 56.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Reprojection with rioxarray+dask

Finally we will be using rioxarray + dask. We create a function called *reproj_rioxarray_dask* that combines reprojecting and saving a new file.

Since we are using dask in this case, we have chunked our array by 2048 x 2048, which is generally a good compromise.

In [72]:
dst_crs = "EPSG:4326"

dst_path = "../outputs/usecase_rxr_dask_reproj.tif"

def reproj_rioxarray_dask():
    client = Client()
    ds_rxr_chunked = rxr.open_rasterio(raster_path, chunks={'x' : 2048, 'y' : 2048 })
    rxr_reproj_dask = ds_rxr_chunked.rio.reproject(dst_crs)
    rxr_reproj_dask.rio.to_raster(dst_path)
    client.close()

Here if the result if we call the new function :

In [73]:
%%time
mem_rioxarray_dask = memory_usage((reproj_rioxarray_dask,))
print(f"RAM max : {max(mem_rioxarray_dask):.1f} MB")

c:\Users\cbernaert\AppData\Local\anaconda3\envs\open_source_rasters_comparison\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 55876 instead
  warnings.warn(


RAM max : 2794.4 MB
CPU times: total: 3.56 s
Wall time: 6.8 s
